
# 11 — Ground Truth and Quality Validation

## Overview

This notebook constructs and validates the attack Ground Truth for the Mubsir dataset using the approved attacker assignments, daily attack schedules, and verified injection outputs from Notebooks 08, 09, and 10.

The Ground Truth records **who performed the attack, when it occurred, which channels were involved, and which approved scenario dimensions were applied**.

## Objectives

1. Load and verify the approved scenario assignments, attack schedules, and injection outputs.
2. Construct two complementary Ground Truth datasets:
   - **User-Day Ground Truth:** Records attack days, scenario details, channel rotation, synchronization, and daily injection summaries.
   - **Channel-Day Ground Truth:** Records individual injected channels, event increments, and execution evidence.
3. Preserve relevant attack dimensions, including scenario type, attack duration, channel configuration, switching level, and stealth level.
4. Verify that every Ground Truth record corresponds to an actual, authorized injection.
5. Validate consistency between both Ground Truth datasets and the original attack schedule.
6. Export the validated Ground Truth datasets with quality reports and integrity checks.

## Expected Outputs

- `ground_truth_user_days_r62.parquet`
- `ground_truth_channel_days_r62.parquet`
- Quality assurance reports and export manifest.

## Scope and Constraints

- **Attack-only Ground Truth:** Includes only actual injected attack records.
- **No classification labels (`y`):** Labels will be generated in Notebook 12 when preparing the final Behavioral Features dataset.
- **No reinjection:** Existing injected User-Day data remains unchanged.
- **No baseline recalculation:** Historical behavioral references remain immutable.
- **No feature engineering:** Behavioral features are handled separately in Notebook 12.
- **No redundant User-Level Ground Truth:** Approved attacker-level information remains available from Notebook 08.

## Expected Outcome

Two validated, traceable Ground Truth datasets that accurately represent the executed channel-rotating, low-and-slow insider data exfiltration scenarios and support subsequent feature engineering, model training, and evaluation.



## 11.1 — Input Inventory & Integrity Verification

### Objective

Establish a reliable, read-only foundation for constructing attack Ground Truth using the approved scenario assignments, daily attack schedule, and validated injection outputs.

### Required Sources

- **Notebook 08:** Approved attacker scenarios and assigned scenario dimensions.
- **Notebook 09:** Approved daily attack schedule and channel switching information.
- **Notebook 10:** Corrected injected User-Day, injection execution log, and export manifest.

### Validation Scope

1. Resolve source paths and verify required files exist.
2. Verify all Notebook 10 artifact SHA-256 checksums against its export manifest.
3. Load the required Parquet datasets without modifying them.
4. Inspect dataset shapes, key columns, schemas, and nulls.
5. Verify the expected attacker population and injection record counts.
6. Confirm source readiness for Ground Truth construction.

### Ground Truth Design

Two attack-only datasets will be constructed:

- `ground_truth_user_days_r62.parquet`
- `ground_truth_channel_days_r62.parquet`

Neither dataset will contain an `y` target column. Binary classification labels will be created later in Notebook 12, using the User-Day Ground Truth as the authoritative reference.

### Restrictions

- No re-injection or adjustment of event counts.
- No recalculation of behavioral baselines.
- No generation of model features or classification labels.
- No modification of existing source files.

### Expected Outcome

Verified source datasets and an approved input inventory for the subsequent Ground Truth construction stages.


In [1]:

# ============================================================
# NOTEBOOK 11 — SECTION 11.1
# INPUT INVENTORY & INTEGRITY VERIFICATION
# READ-ONLY — NO EXPORTS
# ============================================================

from pathlib import Path
import hashlib
import json
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# STEP 1 — Resolve project root
# ------------------------------------------------------------

def find_project_root_11(start):
    current = Path(start).resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "reports").is_dir()
            and (
                candidate / "reports"
                / "10_Attack_Injection_Pipeline"
            ).is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Cannot locate New_Attack_Simulation project root."
    )


ROOT_11 = find_project_root_11(Path.cwd())
REPORTS_11 = ROOT_11 / "reports"

REPORT_08_11 = REPORTS_11 / "08_Attacker_Assignment"
REPORT_09_11 = REPORTS_11 / "09_PreInjection_Dry_Run"
REPORT_10_11 = REPORTS_11 / "10_Attack_Injection_Pipeline"

def require_11(condition, message):
    if not bool(condition):
        raise AssertionError(f"11.1 FAILED: {message}")


# ------------------------------------------------------------
# STEP 2 — Resolve and verify Notebook 10 outputs
# ------------------------------------------------------------

manifest_path_11 = (
    REPORT_10_11 / "export_manifest.json"
)

require_11(
    manifest_path_11.is_file(),
    "Notebook 10 export manifest is missing."
)

manifest_11 = json.loads(
    manifest_path_11.read_text(encoding="utf-8")
)

require_11(
    manifest_11.get("status")
    == "INDEPENDENT_VALIDATION_APPROVED",
    "Notebook 10 export is not approved."
)

require_11(
    manifest_11.get("validation", {}).get(
        "independent_checks_passed"
    ) == 43,
    "Notebook 10 independent QA is incomplete."
)


def sha256_11(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(8 * 1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


artifact_checks_11 = []

for filename, metadata in manifest_11["artifacts"].items():

    path = REPORT_10_11 / filename
    exists = path.is_file()

    valid = (
        exists
        and path.stat().st_size == metadata["size_bytes"]
        and sha256_11(path) == metadata["sha256"]
    )

    artifact_checks_11.append({
        "artifact": filename,
        "status": "PASS" if valid else "FAIL",
    })

artifact_checks_11 = pd.DataFrame(artifact_checks_11)

require_11(
    artifact_checks_11["status"].eq("PASS").all(),
    "One or more Notebook 10 artifacts failed integrity checks."
)


# ------------------------------------------------------------
# STEP 3 — Load required Notebook 10 source datasets
# ------------------------------------------------------------

injected_user_day_11 = pd.read_parquet(
    REPORT_10_11 / "user_day_injected_counts_r62.parquet"
)

injection_execution_log_11 = pd.read_parquet(
    REPORT_10_11 / "injection_execution_log_r62.parquet"
)

injection_plan_11 = pd.read_parquet(
    REPORT_10_11 / "daily_injection_plan_r62.parquet"
)

require_11(
    injected_user_day_11.shape[0] == 1_394_010
    and injected_user_day_11.shape[1] == 34
    and injected_user_day_11["user"].nunique() == 4000,
    "Injected User-Day population/schema mismatch."
)

require_11(
    len(injection_execution_log_11) == 10_124
    and len(injection_plan_11) == 10_124,
    "Injection records are incomplete."
)


# ------------------------------------------------------------
# STEP 4 — Load Notebook 09 daily schedule
# ------------------------------------------------------------

schedule_path_11 = (
    REPORT_09_11 / "daily_attack_schedule_r62.parquet"
)

require_11(
    schedule_path_11.is_file(),
    "Notebook 09 daily schedule is missing."
)

daily_schedule_11 = pd.read_parquet(schedule_path_11)

require_11(
    len(daily_schedule_11) == 10_750
    and daily_schedule_11["user"].nunique() == 400,
    "Notebook 09 schedule dimensions mismatch."
)

require_11(
    int(daily_schedule_11["is_active_day"].sum()) == 9080
    and int(daily_schedule_11["is_forced_offday"].sum()) == 1670,
    "Notebook 09 active/off-day counts mismatch."
)


# ------------------------------------------------------------
# STEP 5 — Discover Notebook 08 approved assignments
#
# Do not guess the final filename or choose a source
# automatically when multiple candidates are available.
# ------------------------------------------------------------

assignment_candidates_11 = sorted({
    p.resolve()
    for p in REPORTS_11.rglob("*.parquet")
    if "08" in p.parent.name
    and (
        "assign" in p.name.lower()
        or "approved" in p.name.lower()
        or "scenario" in p.name.lower()
    )
})

assignment_inventory_11 = []

for path in assignment_candidates_11:
    try:
        import pyarrow.parquet as pq

        metadata = pq.read_metadata(path)

        assignment_inventory_11.append({
            "filename": path.name,
            "rows": metadata.num_rows,
            "columns": metadata.num_columns,
            "path": str(path),
        })

    except Exception as exc:
        assignment_inventory_11.append({
            "filename": path.name,
            "rows": None,
            "columns": None,
            "path": str(path),
        })

assignment_inventory_11 = pd.DataFrame(
    assignment_inventory_11,
    columns=["filename", "rows", "columns", "path"],
)


# ------------------------------------------------------------
# STEP 6 — Inspect schemas required for Notebook 11
# ------------------------------------------------------------

datasets_11 = {
    "Injected User-Day": injected_user_day_11,
    "Injection Execution Log": injection_execution_log_11,
    "Injection Plan": injection_plan_11,
    "Daily Schedule": daily_schedule_11,
}

input_inventory_11 = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(frame),
        "columns": len(frame.columns),
        "users": (
            frame["user"].nunique()
            if "user" in frame.columns
            else None
        ),
    }
    for name, frame in datasets_11.items()
])

input_schemas_11 = {
    name: frame.columns.tolist()
    for name, frame in datasets_11.items()
}


# ------------------------------------------------------------
# STEP 7 — Verify essential linking keys
# ------------------------------------------------------------

for name, frame in datasets_11.items():
    require_11(
        {"user", "day"}.issubset(frame.columns),
        f"{name}: missing user/day keys."
    )

    require_11(
        frame[["user", "day"]].notna().all().all(),
        f"{name}: null user/day keys."
    )

require_11(
    {"channel", "count_added"}.issubset(
        injection_execution_log_11.columns
    ),
    "Execution log is missing channel/count evidence."
)

require_11(
    {
        "is_active_day",
        "is_forced_offday",
        "scheduled_channel_mask",
    }.issubset(daily_schedule_11.columns),
    "Daily schedule is missing required attack metadata."
)


# ------------------------------------------------------------
# STEP 8 — Print concise inventory
# ------------------------------------------------------------

print("\n11.1 — VERIFIED INPUT DATASETS")
display(input_inventory_11)

print("\nNOTEBOOK 08 — APPROVED ASSIGNMENT FILE CANDIDATES")
display(assignment_inventory_11)

print("\nNOTEBOOK 10 — MANIFEST INTEGRITY")
print(
    f"{artifact_checks_11['status'].eq('PASS').sum()}"
    f"/{len(artifact_checks_11)} artifacts verified"
)

print("\nSOURCE COLUMN INVENTORY")

for name, columns in input_schemas_11.items():
    print(f"\n{name} ({len(columns)} columns)")
    print(", ".join(columns))

print("\n" + "=" * 55)
print("11.1 — INPUT INVENTORY")
print("Notebook 10 integrity : PASS")
print("Notebook 09 schedule  : PASS")
print("User-Day population   : PASS")
print("Execution action keys : PASS")
print("Existing files changed: NO")
print("Notebook 08 assignment: CANDIDATE DISCOVERY")
print("=" * 55)



11.1 — VERIFIED INPUT DATASETS


,dataset,rows,columns,users
0,Injected User-Day,1394010,34,4000
1,Injection Execution Log,10124,18,400
2,Injection Plan,10124,19,400
3,Daily Schedule,10750,46,400



NOTEBOOK 08 — APPROVED ASSIGNMENT FILE CANDIDATES


,filename,rows,columns,path
0,approved_attacker_scenarios_r62.parquet,400,23,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...



NOTEBOOK 10 — MANIFEST INTEGRITY
14/14 artifacts verified

SOURCE COLUMN INVENTORY

Injected User-Day (34 columns)
user, day, day_of_week, is_weekend, email_event_count, email_send_count, email_view_count, external_email_count, internal_email_count, total_sent_email_bytes, external_email_bytes, emails_with_attachments, attachment_count, unique_external_recipients, unique_external_domains, file_event_count, file_copy_count, file_open_count, file_write_count, file_delete_count, usb_copy_count, from_removable_count, unique_usb_files, unique_usb_extensions, http_event_count, www_visit_count, www_download_count, www_upload_count, unique_domains, unique_upload_domains, logon_count, logoff_count, first_logon, last_logoff

Injection Execution Log (18 columns)
user, day, scenario_option_id, workday_index, channel, source_count_field, channel_configuration, primary_channel, scheduled_channel_set, sync_type, is_switch, formula_id, stealth_level, stealth_multiplier, baseline_p95, count_before, co


## 11.2 — Approved Scenario & Execution Evidence Reconciliation

### Objective

Establish a verified relationship between the approved attacker assignments, daily attack schedule, and actual channel-level injection evidence.

### Validation

1. Identify and load the uniquely approved Notebook 08 assignment file.
2. Verify exactly 400 unique approved attackers.
3. Match the approved scenario assignments to all scheduled attacker-days.
4. Confirm that every execution record belongs to an authorized active day.
5. Reconstruct the executed channel mask independently from the execution log.
6. Verify that executed channels exactly match the approved daily schedule.
7. Verify that all 9,080 active days have injection evidence and all 1,670 forced off-days have none.
8. Reconcile relevant scenario and execution metadata.

### Expected Outcome

An approved, read-only evidence mapping that can be used to construct:

- User-Day Ground Truth.
- Channel-Day Ground Truth.

No Ground Truth files are exported at this stage.


In [2]:

# ============================================================
# NOTEBOOK 11 — SECTION 11.2
# APPROVED SCENARIO & EXECUTION EVIDENCE RECONCILIATION
# READ-ONLY — NO EXPORTS
# ============================================================

from functools import reduce
from operator import or_
import numpy as np
import pandas as pd
from IPython.display import display


def require_112(condition, message):
    if not bool(condition):
        raise AssertionError(f"11.2 FAILED: {message}")


qa_rows_112 = []

def check_112(condition, description):
    require_112(condition, description)
    qa_rows_112.append({
        "check": description,
        "status": "PASS",
    })


# ------------------------------------------------------------
# STEP 1 — Load the unique approved Notebook 08 source
# ------------------------------------------------------------

approved_paths_112 = [
    p for p in assignment_candidates_11
    if p.name == "approved_attacker_scenarios_r62.parquet"
]

check_112(
    len(approved_paths_112) == 1,
    "Unique Notebook 08 approved assignment source"
)

assignment_source_112 = approved_paths_112[0]

approved_assignments_11 = pd.read_parquet(
    assignment_source_112
)

check_112(
    len(approved_assignments_11) == 400
    and approved_assignments_11["user"].nunique() == 400
    and approved_assignments_11["user"].notna().all(),
    "Exactly 400 unique approved attackers"
)

check_112(
    {"user", "scenario_option_id"}.issubset(
        approved_assignments_11.columns
    ),
    "Required assignment identifiers available"
)


# ------------------------------------------------------------
# STEP 2 — Normalize linking keys in working copies
# ------------------------------------------------------------

assign_112 = approved_assignments_11.copy()
schedule_112 = daily_schedule_11.copy()
execution_112 = injection_execution_log_11.copy()

for frame in (assign_112, schedule_112, execution_112):
    frame["user"] = frame["user"].astype("string")

for frame in (schedule_112, execution_112):
    frame["day"] = pd.to_datetime(
        frame["day"]
    ).dt.normalize()

check_112(
    not schedule_112.duplicated(["user", "day"]).any(),
    "Daily schedule User-Day keys unique"
)

check_112(
    not execution_112.duplicated(
        ["user", "day", "channel"]
    ).any(),
    "Execution User-Day-Channel keys unique"
)


# ------------------------------------------------------------
# STEP 3 — Verify schedule against approved assignments
# ------------------------------------------------------------

assignment_fields_112 = [
    c for c in assign_112.columns if c != "user"
]

approved_renamed_112 = assign_112.rename(
    columns={
        c: f"approved__{c}"
        for c in assignment_fields_112
    }
)

schedule_approved_112 = schedule_112.merge(
    approved_renamed_112,
    on="user",
    how="left",
    validate="many_to_one",
    indicator=True,
)

check_112(
    schedule_approved_112["_merge"].eq("both").all()
    and set(schedule_112["user"])
        == set(assign_112["user"]),
    "Every scheduled attacker has an approved assignment"
)

schedule_approved_112 = schedule_approved_112.drop(
    columns="_merge"
)

assignment_compare_fields_112 = [
    "scenario_option_id",
    "channel_configuration",
    "switching_level",
    "stealth_level",
    "stealth_multiplier",
    "window_workdays",
    "window_start",
    "window_end",
]

compared_assignment_fields_112 = []

for field in assignment_compare_fields_112:
    approved_col = f"approved__{field}"

    if (
        field not in schedule_approved_112.columns
        or approved_col not in schedule_approved_112.columns
    ):
        continue

    left = schedule_approved_112[field]
    right = schedule_approved_112[approved_col]

    if field in ("window_start", "window_end"):
        left = pd.to_datetime(left)
        right = pd.to_datetime(right)

    matches = (
        left.eq(right).fillna(False)
        | (left.isna() & right.isna())
    )

    check_112(
        matches.all(),
        f"Approved assignment matches schedule: {field}"
    )

    compared_assignment_fields_112.append(field)


# ------------------------------------------------------------
# STEP 4 — Verify scheduled day categories
# ------------------------------------------------------------

active_112 = schedule_112["is_active_day"].eq(True)
offday_112 = schedule_112["is_forced_offday"].eq(True)

check_112(
    len(schedule_112) == 10_750
    and int(active_112.sum()) == 9_080
    and int(offday_112.sum()) == 1_670,
    "Approved schedule population and day categories"
)

check_112(
    not (active_112 & offday_112).any()
    and (active_112 | offday_112).all(),
    "Active and forced off-days are mutually exclusive"
)


# ------------------------------------------------------------
# STEP 5 — Connect actual execution to scheduled days
# ------------------------------------------------------------

schedule_fields_112 = [
    c for c in schedule_approved_112.columns
    if c not in ("user", "day")
]

schedule_renamed_112 = schedule_approved_112.rename(
    columns={
        c: f"schedule__{c}"
        for c in schedule_fields_112
    }
)

execution_evidence_11 = execution_112.merge(
    schedule_renamed_112,
    on=["user", "day"],
    how="left",
    validate="many_to_one",
    indicator=True,
)

check_112(
    execution_evidence_11["_merge"].eq("both").all(),
    "Every execution action matches a scheduled User-Day"
)

execution_evidence_11 = execution_evidence_11.drop(
    columns="_merge"
)

check_112(
    execution_evidence_11[
        "schedule__is_active_day"
    ].eq(True).all(),
    "Every executed action occurs on an active day"
)

check_112(
    execution_evidence_11[
        "schedule__is_forced_offday"
    ].eq(False).all(),
    "No executed action occurs on a forced off-day"
)


# ------------------------------------------------------------
# STEP 6 — Verify exact scheduled-versus-executed channels
# ------------------------------------------------------------

CHANNEL_MASK_112 = {
    "EMAIL": 1,
    "USB": 2,
    "HTTP": 4,
}

check_112(
    execution_112["channel"].isin(
        CHANNEL_MASK_112
    ).all(),
    "All executed channels are approved"
)

execution_112["channel_bit_112"] = (
    execution_112["channel"].map(CHANNEL_MASK_112)
)

executed_days_112 = (
    execution_112
    .groupby(["user", "day"], as_index=False)
    .agg(
        executed_channel_mask=(
            "channel_bit_112",
            lambda x: reduce(
                or_, (int(v) for v in x), 0
            )
        ),
        executed_channel_count=("channel", "size"),
        executed_event_count=("count_added", "sum"),
    )
)

schedule_execution_112 = schedule_112.merge(
    executed_days_112,
    on=["user", "day"],
    how="left",
    validate="one_to_one",
)

scheduled_mask_112 = pd.to_numeric(
    schedule_execution_112["scheduled_channel_mask"],
    errors="raise",
).astype("int64")

executed_mask_112 = (
    schedule_execution_112["executed_channel_mask"]
    .fillna(0)
    .astype("int64")
)

check_112(
    (scheduled_mask_112 == executed_mask_112).all(),
    "Executed channel masks exactly match the schedule"
)

check_112(
    (
        schedule_execution_112.loc[
            active_112.to_numpy(),
            "executed_channel_count"
        ].notna().all()
        and schedule_execution_112.loc[
            offday_112.to_numpy(),
            "executed_channel_count"
        ].isna().all()
    ),
    "All active days executed; all off-days have no execution"
)

check_112(
    len(executed_days_112) == 9_080
    and len(execution_112) == 10_124,
    "Executed User-Day and channel-action totals"
)


# ------------------------------------------------------------
# STEP 7 — Reconcile execution metadata with daily schedule
# ------------------------------------------------------------

execution_compare_fields_112 = [
    "scenario_option_id",
    "channel_configuration",
    "primary_channel",
    "scheduled_channel_set",
    "sync_type",
    "is_switch",
    "stealth_level",
    "stealth_multiplier",
]

for field in execution_compare_fields_112:
    schedule_col = f"schedule__{field}"

    if (
        field not in execution_evidence_11.columns
        or schedule_col not in execution_evidence_11.columns
    ):
        continue

    left = execution_evidence_11[field]
    right = execution_evidence_11[schedule_col]

    matches = (
        left.eq(right).fillna(False)
        | (left.isna() & right.isna())
    )

    check_112(
        matches.all(),
        f"Execution metadata matches schedule: {field}"
    )

check_112(
    execution_112["count_added"].notna().all()
    and execution_112["count_added"].gt(0).all()
    and int(execution_112["count_added"].sum()) == 29_007,
    "All executed increments positive; total is 29,007"
)


# ------------------------------------------------------------
# STEP 8 — Final evidence mapping checks
# ------------------------------------------------------------

check_112(
    len(execution_evidence_11) == 10_124
    and len(schedule_execution_112) == 10_750,
    "Evidence joins preserve exact source row counts"
)

check_112(
    execution_evidence_11[
        ["user", "day", "channel"]
    ].notna().all().all()
    and not execution_evidence_11.duplicated(
        ["user", "day", "channel"]
    ).any(),
    "Validated channel-level evidence has unique keys"
)

check_112(
    schedule_execution_112[
        ["user", "day"]
    ].notna().all().all()
    and not schedule_execution_112.duplicated(
        ["user", "day"]
    ).any(),
    "Validated daily evidence has unique keys"
)

scenario_execution_qa_11 = pd.DataFrame(qa_rows_112)

require_112(
    scenario_execution_qa_11["status"].eq("PASS").all(),
    "Scenario/execution evidence reconciliation incomplete"
)

scenario_execution_approved_11 = True


# ------------------------------------------------------------
# STEP 9 — Concise result
# ------------------------------------------------------------

reconciliation_summary_112 = pd.DataFrame([
    {
        "measure": "Approved attackers",
        "value": len(assign_112),
    },
    {
        "measure": "Scheduled User-Days",
        "value": len(schedule_112),
    },
    {
        "measure": "Active attack User-Days",
        "value": len(executed_days_112),
    },
    {
        "measure": "Forced off-days without injection",
        "value": int(offday_112.sum()),
    },
    {
        "measure": "Verified channel-actions",
        "value": len(execution_evidence_11),
    },
    {
        "measure": "Verified injected events",
        "value": int(execution_112["count_added"].sum()),
    },
])

print("\n11.2 — EVIDENCE RECONCILIATION SUMMARY")
display(reconciliation_summary_112)

print("\n" + "=" * 55)
print("11.2 — APPROVED SCENARIO & EXECUTION EVIDENCE")
print("Assignment source      :", assignment_source_112.name)
print("Assignment fields tested:",
      ", ".join(compared_assignment_fields_112))
print(
    "Validation             :",
    f"{len(scenario_execution_qa_11)}/"
    f"{len(scenario_execution_qa_11)} PASS"
)
print("Schedule reconciliation: PASS")
print("Channel-mask agreement : EXACT MATCH")
print("Original files modified: NO")
print("Ground Truth exported  : NO")
print("Status                 : APPROVED")
print("=" * 55)



11.2 — EVIDENCE RECONCILIATION SUMMARY


,measure,value
0,Approved attackers,400
1,Scheduled User-Days,10750
2,Active attack User-Days,9080
3,Forced off-days without injection,1670
4,Verified channel-actions,10124
5,Verified injected events,29007



11.2 — APPROVED SCENARIO & EXECUTION EVIDENCE
Assignment source      : approved_attacker_scenarios_r62.parquet
Assignment fields tested: scenario_option_id, channel_configuration, switching_level, stealth_level, stealth_multiplier, window_workdays, window_start, window_end
Validation             : 35/35 PASS
Schedule reconciliation: PASS
Channel-mask agreement : EXACT MATCH
Original files modified: NO
Ground Truth exported  : NO
Status                 : APPROVED



## 11.3 — Channel-Day Ground Truth Construction

### Objective

Construct an execution-based, channel-level attack Ground Truth dataset using the approved and reconciled injection evidence from Section 11.2.

### Record Granularity

One record per unique `(user, day, channel)` where an authorized injection was actually executed.

### Ground Truth Contents

- **Identity:** User, day, channel, and scenario identifier.
- **Execution:** Original event count, injected increment, and final event count.
- **Scenario:** Channel configuration, stealth level, stealth multiplier, and injection formula.
- **Temporal context:** Attack-window dates, workday index, and active-day index.
- **Rotation:** Primary channel, scheduled channel set, and switching indicator.
- **Synchronization:** Same-day channel synchronization type.
- **Traceability:** Historical P95 reference used by the approved injection formula.

### Validation

1. Preserve all 10,124 verified channel-actions.
2. Require unique `(user, day, channel)` keys.
3. Confirm exactly 400 attack users.
4. Require positive injected event increments.
5. Verify `count_after = count_before + count_added`.
6. Confirm the exact aggregate of 29,007 injected events.
7. Verify channel and daily consistency against the approved schedule.
8. Ensure no forced off-days, nonattackers, or unapproved channels appear.
9. Confirm no `y` classification target or behavioral features are included.

### Expected Outcome

An in-memory `ground_truth_channel_days_11` dataset, ready for independent validation and subsequent construction of the User-Day Ground Truth.

No files will be exported in this section.


In [3]:

# ============================================================
# NOTEBOOK 11 — SECTION 11.3
# CHANNEL-DAY GROUND TRUTH CONSTRUCTION
# READ-ONLY SOURCES — NO EXPORTS
# ============================================================

import pandas as pd
from pandas.testing import assert_frame_equal
from IPython.display import display


# ------------------------------------------------------------
# STEP 1 — Prerequisites
# ------------------------------------------------------------

def require_113(condition, message):
    if not bool(condition):
        raise AssertionError(f"11.3 FAILED: {message}")


require_113(
    scenario_execution_approved_11 is True,
    "Section 11.2 reconciliation is not approved."
)

qa_rows_113 = []


def check_113(condition, description):
    require_113(condition, description)
    qa_rows_113.append({
        "check": description,
        "status": "PASS",
    })


# ------------------------------------------------------------
# STEP 2 — Create Channel-Day Ground Truth
#
# Source of truth:
# Actual injection execution log from Notebook 10.
#
# Add approved scheduling context from Notebook 09.
# ------------------------------------------------------------

ground_truth_channel_days_11 = (
    injection_execution_log_11.copy(deep=True)
)

ground_truth_channel_days_11["user"] = (
    ground_truth_channel_days_11["user"].astype("string")
)

ground_truth_channel_days_11["day"] = pd.to_datetime(
    ground_truth_channel_days_11["day"]
).dt.normalize()


# Schedule-only fields needed for Ground Truth context.
schedule_context_fields_113 = [
    "active_day_index",
    "window_workdays",
    "window_start",
    "window_end",
    "switching_level",
    "forced_offdays",
    "scheduled_channel_mask",
    "is_active_day",
    "is_forced_offday",
]

required_schedule_columns_113 = [
    "user",
    "day",
    *schedule_context_fields_113,
]

check_113(
    set(required_schedule_columns_113).issubset(
        daily_schedule_11.columns
    ),
    "Required scheduling context is available"
)

schedule_context_113 = daily_schedule_11[
    required_schedule_columns_113
].copy()

schedule_context_113["user"] = (
    schedule_context_113["user"].astype("string")
)

schedule_context_113["day"] = pd.to_datetime(
    schedule_context_113["day"]
).dt.normalize()

for col in ("window_start", "window_end"):
    schedule_context_113[col] = pd.to_datetime(
        schedule_context_113[col]
    ).dt.normalize()

check_113(
    not schedule_context_113.duplicated(
        ["user", "day"]
    ).any(),
    "Scheduling context has unique User-Day keys"
)

ground_truth_channel_days_11 = (
    ground_truth_channel_days_11.merge(
        schedule_context_113,
        on=["user", "day"],
        how="left",
        validate="many_to_one",
        indicator=True,
    )
)

check_113(
    ground_truth_channel_days_11["_merge"]
    .eq("both").all(),
    "All channel actions have scheduling context"
)

ground_truth_channel_days_11.drop(
    columns="_merge",
    inplace=True,
)


# ------------------------------------------------------------
# STEP 3 — Verify execution identity and population
# ------------------------------------------------------------

check_113(
    len(ground_truth_channel_days_11) == 10_124,
    "Exactly 10,124 executed channel-actions"
)

check_113(
    ground_truth_channel_days_11["user"].nunique()
    == 400,
    "Exactly 400 distinct attack users"
)

check_113(
    ground_truth_channel_days_11[
        ["user", "day", "channel"]
    ].notna().all().all()
    and not ground_truth_channel_days_11.duplicated(
        ["user", "day", "channel"]
    ).any(),
    "Unique and non-null User-Day-Channel keys"
)

check_113(
    set(ground_truth_channel_days_11["user"])
    == set(approved_assignments_11["user"].astype("string")),
    "Ground Truth users exactly match approved attackers"
)


# ------------------------------------------------------------
# STEP 4 — Verify execution evidence
# ------------------------------------------------------------

numeric_fields_113 = [
    "count_before",
    "count_added",
    "count_after",
    "baseline_p95",
]

check_113(
    ground_truth_channel_days_11[
        numeric_fields_113
    ].notna().all().all(),
    "All count and reference values are present"
)

check_113(
    (
        ground_truth_channel_days_11["count_added"]
        > 0
    ).all(),
    "Every record represents a positive injection"
)

check_113(
    (
        ground_truth_channel_days_11["count_before"]
        + ground_truth_channel_days_11["count_added"]
        == ground_truth_channel_days_11["count_after"]
    ).all(),
    "Every before/add/after count reconciles"
)

check_113(
    int(
        ground_truth_channel_days_11[
            "count_added"
        ].sum()
    ) == 29_007,
    "Exactly 29,007 injected attack events"
)


# ------------------------------------------------------------
# STEP 5 — Verify scheduling and channel authorization
# ------------------------------------------------------------

check_113(
    ground_truth_channel_days_11[
        "is_active_day"
    ].eq(True).all(),
    "Every Ground Truth record belongs to an active day"
)

check_113(
    ground_truth_channel_days_11[
        "is_forced_offday"
    ].eq(False).all(),
    "No Ground Truth record belongs to a forced off-day"
)

channel_bits_113 = {
    "EMAIL": 1,
    "USB": 2,
    "HTTP": 4,
}

check_113(
    ground_truth_channel_days_11[
        "channel"
    ].isin(channel_bits_113).all(),
    "Only approved EMAIL, USB and HTTP channels appear"
)

actual_channel_bits_113 = (
    ground_truth_channel_days_11["channel"]
    .map(channel_bits_113)
    .astype("int64")
)

scheduled_masks_113 = (
    ground_truth_channel_days_11[
        "scheduled_channel_mask"
    ].astype("int64")
)

check_113(
    (
        scheduled_masks_113
        .to_numpy()
        & actual_channel_bits_113.to_numpy()
    == actual_channel_bits_113.to_numpy()).all(),
    "Every injected channel is authorized by its daily mask"
)

check_113(
    ground_truth_channel_days_11[
        "scenario_option_id"
    ].notna().all()
    and ground_truth_channel_days_11[
        "stealth_level"
    ].notna().all()
    and ground_truth_channel_days_11[
        "formula_id"
    ].notna().all(),
    "Required scenario and formula identifiers are present"
)


# ------------------------------------------------------------
# STEP 6 — Verify exact execution-log preservation
# ------------------------------------------------------------

source_execution_113 = (
    injection_execution_log_11.copy()
)

source_execution_113["user"] = (
    source_execution_113["user"].astype("string")
)

source_execution_113["day"] = pd.to_datetime(
    source_execution_113["day"]
).dt.normalize()

actual_execution_113 = (
    ground_truth_channel_days_11[
        source_execution_113.columns
    ].copy()
)

assert_frame_equal(
    source_execution_113.reset_index(drop=True),
    actual_execution_113.reset_index(drop=True),
    check_dtype=False,
    check_exact=True,
)

check_113(
    True,
    "All 18 execution-log fields preserved exactly"
)


# ------------------------------------------------------------
# STEP 7 — Validate channel and User-Day totals
# ------------------------------------------------------------

channel_summary_113 = (
    ground_truth_channel_days_11
    .groupby("channel", as_index=False)
    .agg(
        channel_actions=("user", "size"),
        injected_events=("count_added", "sum"),
    )
)

expected_channel_totals_113 = {
    "EMAIL": (6703, 17791),
    "USB": (1935, 4920),
    "HTTP": (1486, 6296),
}

channel_totals_match_113 = all(
    (
        int(
            channel_summary_113.loc[
                channel_summary_113["channel"].eq(channel),
                "channel_actions",
            ].iloc[0]
        ),
        int(
            channel_summary_113.loc[
                channel_summary_113["channel"].eq(channel),
                "injected_events",
            ].iloc[0]
        ),
    ) == expected
    for channel, expected in expected_channel_totals_113.items()
)

check_113(
    channel_totals_match_113,
    "Exact channel-level action and event totals"
)

unique_attack_days_113 = (
    ground_truth_channel_days_11[
        ["user", "day"]
    ]
    .drop_duplicates()
)

check_113(
    len(unique_attack_days_113) == 9_080,
    "Exactly 9,080 distinct attacked User-Days"
)

check_113(
    "y" not in ground_truth_channel_days_11.columns,
    "Ground Truth contains no model target y"
)


# ------------------------------------------------------------
# STEP 8 — Final QA and concise output
# ------------------------------------------------------------

channel_ground_truth_qa_11 = pd.DataFrame(
    qa_rows_113
)

require_113(
    channel_ground_truth_qa_11[
        "status"
    ].eq("PASS").all(),
    "Channel-Day Ground Truth QA incomplete"
)

channel_ground_truth_approved_11 = True

print("\n11.3 — CHANNEL-DAY GROUND TRUTH")
display(channel_summary_113)

print("\n" + "=" * 56)
print("11.3 — CHANNEL-DAY GROUND TRUTH VALIDATION")
print("=" * 56)
print(
    "Channel-Day records     :",
    len(ground_truth_channel_days_11)
)
print(
    "Unique attack users      :",
    ground_truth_channel_days_11["user"].nunique()
)
print(
    "Unique attacked User-Days:",
    len(unique_attack_days_113)
)
print(
    "Injected attack events  :",
    int(
        ground_truth_channel_days_11[
            "count_added"
        ].sum()
    )
)
print(
    "Ground Truth columns     :",
    len(ground_truth_channel_days_11.columns)
)
print(
    "QA checks                :",
    f"{len(channel_ground_truth_qa_11)}/"
    f"{len(channel_ground_truth_qa_11)} PASS"
)
print("Source files modified    : NO")
print("Classification y added   : NO")
print("Files exported           : NO")
print("Status                   : APPROVED")
print("=" * 56)



11.3 — CHANNEL-DAY GROUND TRUTH


,channel,channel_actions,injected_events
0,EMAIL,6703,17791
1,HTTP,1486,6296
2,USB,1935,4920



11.3 — CHANNEL-DAY GROUND TRUTH VALIDATION
Channel-Day records     : 10124
Unique attack users      : 400
Unique attacked User-Days: 9080
Injected attack events  : 29007
Ground Truth columns     : 27
QA checks                : 20/20 PASS
Source files modified    : NO
Classification y added   : NO
Files exported           : NO
Status                   : APPROVED



## 11.4 — User-Day Ground Truth Construction

### Objective

Construct the attack-only User-Day Ground Truth by combining verified channel-level injection evidence with the approved daily schedule and attacker scenario assignments.

### Record Granularity

One unique `(user, day)` record for each day on which an authorized attack injection actually occurred.

### Ground Truth Contents

- **Identity:** User, date, and approved scenario identifier.
- **Scenario dimensions:** Attack duration, channel configuration, switching level, stealth level, formula, and other approved assignment metadata.
- **Attack timeline:** Window start/end, workday index, and active-day index.
- **Channel rotation:** Primary channel, switching indicator, scheduled and executed channels.
- **Same-day synchronization:** Synchronization type and executed channel count.
- **Execution evidence:** Total daily injected events and per-channel injected event counts.

### Validation Requirements

1. Preserve exactly 9,080 attacked User-Days across 400 approved attackers.
2. Require unique and non-null `(user, day)` keys.
3. Preserve all approved attacker-level scenario dimensions.
4. Independently aggregate the 10,124 Channel-Day records.
5. Verify exact daily channel-mask and event-count consistency.
6. Verify the expected single-, two-, and three-channel distributions.
7. Confirm that forced off-days and unapproved users are excluded.
8. Preserve execution-based switching and synchronization metadata.
9. Exclude the `y` classification target and derived behavioral features.
10. Keep all original and previously exported datasets unchanged.

### Expected Outcome

A validated in-memory `ground_truth_user_days_11` dataset containing complete attack-day evidence and approved scenario context, ready for final cross-level quality validation.

No files are exported in this section.


In [4]:

# ============================================================
# NOTEBOOK 11 — SECTION 11.4
# USER-DAY GROUND TRUTH CONSTRUCTION
# READ-ONLY SOURCES — NO EXPORTS
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# STEP 1 — Prerequisites and QA helpers
# ------------------------------------------------------------

def require_114(condition, message):
    if not bool(condition):
        raise AssertionError(f"11.4 FAILED: {message}")


require_114(
    channel_ground_truth_approved_11 is True
    and scenario_execution_approved_11 is True,
    "Sections 11.2 and 11.3 must be approved first."
)

qa_rows_114 = []


def check_114(condition, description):
    require_114(condition, description)
    qa_rows_114.append({
        "check": description,
        "status": "PASS",
    })


CHANNEL_BITS_114 = {
    "EMAIL": 1,
    "USB": 2,
    "HTTP": 4,
}


# ------------------------------------------------------------
# STEP 2 — Prepare validated source copies
# ------------------------------------------------------------

channel_114 = ground_truth_channel_days_11.copy()
schedule_114 = daily_schedule_11.copy()
assignment_114 = approved_assignments_11.copy()

for frame in (channel_114, schedule_114, assignment_114):
    frame["user"] = frame["user"].astype("string")

for frame in (channel_114, schedule_114):
    frame["day"] = pd.to_datetime(
        frame["day"]
    ).dt.normalize()

check_114(
    not channel_114.duplicated(
        ["user", "day", "channel"]
    ).any(),
    "Channel-Day source keys are unique"
)

check_114(
    not schedule_114.duplicated(
        ["user", "day"]
    ).any(),
    "Schedule User-Day source keys are unique"
)

check_114(
    not assignment_114.duplicated("user").any(),
    "Approved assignment user keys are unique"
)


# ------------------------------------------------------------
# STEP 3 — Independently aggregate actual injections
# ------------------------------------------------------------

channel_114["channel_bit_114"] = (
    channel_114["channel"]
    .map(CHANNEL_BITS_114)
    .astype("int64")
)

daily_counts_114 = (
    channel_114
    .groupby(["user", "day"], as_index=False)
    .agg(
        executed_channel_count=(
            "channel",
            "size"
        ),
        total_injected_events=(
            "count_added",
            "sum"
        ),
        executed_channel_mask=(
            "channel_bit_114",
            lambda values: int(
                np.bitwise_or.reduce(
                    values.to_numpy(dtype=np.int64)
                )
            )
        ),
    )
)

daily_channel_events_114 = (
    channel_114
    .pivot(
        index=["user", "day"],
        columns="channel",
        values="count_added",
    )
    .reindex(columns=["EMAIL", "USB", "HTTP"])
    .fillna(0)
    .astype("int64")
    .rename(columns={
        "EMAIL": "email_injected_events",
        "USB": "usb_injected_events",
        "HTTP": "http_injected_events",
    })
    .reset_index()
)

daily_evidence_114 = daily_counts_114.merge(
    daily_channel_events_114,
    on=["user", "day"],
    how="left",
    validate="one_to_one",
)

check_114(
    len(daily_evidence_114) == 9_080
    and daily_evidence_114["user"].nunique() == 400,
    "Exactly 9,080 attacked User-Days across 400 attackers"
)

check_114(
    (
        daily_evidence_114[
            [
                "email_injected_events",
                "usb_injected_events",
                "http_injected_events",
            ]
        ].sum(axis=1)
        == daily_evidence_114["total_injected_events"]
    ).all(),
    "Daily channel increments sum to the daily total"
)


# ------------------------------------------------------------
# STEP 4 — Create readable executed channel sets
#
# Fixed channel order: EMAIL, USB, HTTP
# ------------------------------------------------------------

def decode_channels_114(mask):
    mask = int(mask)

    return "+".join(
        channel
        for channel, bit in CHANNEL_BITS_114.items()
        if mask & bit
    )


daily_evidence_114["injected_channels"] = (
    daily_evidence_114["executed_channel_mask"]
    .map(decode_channels_114)
)

check_114(
    daily_evidence_114["injected_channels"]
    .notna().all()
    and daily_evidence_114["injected_channels"]
    .ne("").all(),
    "Every attacked day has identified injected channels"
)


# ------------------------------------------------------------
# STEP 5 — Build User-Day context from approved schedule
#
# Only executed active days are included.
# No benign days or forced off-days.
# ------------------------------------------------------------

schedule_fields_114 = [
    "user",
    "day",
    "scenario_option_id",
    "window_workdays",
    "window_start",
    "window_end",
    "channel_configuration",
    "switching_level",
    "stealth_level",
    "stealth_multiplier",
    "forced_offdays",
    "workday_index",
    "active_day_index",
    "primary_channel",
    "primary_channel_mask",
    "scheduled_channel_set",
    "scheduled_channel_mask",
    "scheduled_channel_count",
    "is_switch",
    "sync_type",
    "is_active_day",
    "is_forced_offday",
]

check_114(
    set(schedule_fields_114).issubset(
        schedule_114.columns
    ),
    "All required User-Day schedule fields are present"
)

active_schedule_114 = schedule_114.loc[
    schedule_114["is_active_day"].eq(True),
    schedule_fields_114,
].copy()

for col in ("window_start", "window_end"):
    active_schedule_114[col] = pd.to_datetime(
        active_schedule_114[col]
    ).dt.normalize()

ground_truth_user_days_11 = active_schedule_114.merge(
    daily_evidence_114,
    on=["user", "day"],
    how="outer",
    validate="one_to_one",
    indicator=True,
)

check_114(
    ground_truth_user_days_11["_merge"]
    .eq("both").all(),
    "Executed days exactly match scheduled active days"
)

ground_truth_user_days_11.drop(
    columns="_merge",
    inplace=True,
)


# ------------------------------------------------------------
# STEP 6 — Attach ALL approved assignment dimensions
#
# Reuse shared schedule fields.
# Include any additional approved assignment columns.
# This avoids a separate User-Level Ground Truth file.
# ------------------------------------------------------------

check_114(
    "scenario_option_id" in assignment_114.columns,
    "Approved scenario identifier is available"
)

shared_fields_114 = [
    col
    for col in assignment_114.columns
    if col in ground_truth_user_days_11.columns
    and col != "user"
]

assignment_check_114 = ground_truth_user_days_11.merge(
    assignment_114[
        ["user", *shared_fields_114]
    ].rename(columns={
        col: f"approved__{col}"
        for col in shared_fields_114
    }),
    on="user",
    how="left",
    validate="many_to_one",
)

for field in shared_fields_114:
    left = assignment_check_114[field]
    right = assignment_check_114[
        f"approved__{field}"
    ]

    if field in ("window_start", "window_end"):
        left = pd.to_datetime(left)
        right = pd.to_datetime(right)

    if pd.api.types.is_numeric_dtype(left) and (
        pd.api.types.is_numeric_dtype(right)
    ):
        equal = np.isclose(
            pd.to_numeric(left).to_numpy(dtype=float),
            pd.to_numeric(right).to_numpy(dtype=float),
            rtol=1e-12,
            atol=1e-12,
            equal_nan=True,
        )
    else:
        equal = (
            left.astype("string")
            .eq(right.astype("string"))
            .fillna(False)
            |
            (left.isna() & right.isna())
        ).to_numpy(dtype=bool)

    require_114(
        np.asarray(equal).all(),
        f"Approved scenario mismatch: {field}"
    )

check_114(
    True,
    "All overlapping assignment dimensions match"
)

additional_assignment_fields_114 = [
    col
    for col in assignment_114.columns
    if col not in ground_truth_user_days_11.columns
    and col != "user"
]

if additional_assignment_fields_114:
    ground_truth_user_days_11 = (
        ground_truth_user_days_11.merge(
            assignment_114[
                ["user", *additional_assignment_fields_114]
            ],
            on="user",
            how="left",
            validate="many_to_one",
        )
    )

check_114(
    all(
        field in ground_truth_user_days_11.columns
        for field in assignment_114.columns
    ),
    "All approved assignment dimensions are retained"
)


# ------------------------------------------------------------
# STEP 7 — Validate exact channel execution
# ------------------------------------------------------------

check_114(
    (
        ground_truth_user_days_11[
            "scheduled_channel_mask"
        ].astype("int64")
        ==
        ground_truth_user_days_11[
            "executed_channel_mask"
        ].astype("int64")
    ).all(),
    "Executed and scheduled channel masks match exactly"
)

check_114(
    (
        ground_truth_user_days_11[
            "scheduled_channel_count"
        ].astype("int64")
        ==
        ground_truth_user_days_11[
            "executed_channel_count"
        ].astype("int64")
    ).all(),
    "Executed and scheduled channel counts match exactly"
)

check_114(
    ground_truth_user_days_11[
        "executed_channel_count"
    ].between(1, 3).all(),
    "Every attacked day uses one to three channels"
)


# ------------------------------------------------------------
# STEP 8 — Validate attack-day classifications
# ------------------------------------------------------------

channel_distribution_114 = (
    ground_truth_user_days_11[
        "executed_channel_count"
    ]
    .value_counts()
    .reindex([1, 2, 3], fill_value=0)
    .sort_index()
)

expected_distribution_114 = {
    1: 8073,
    2: 970,
    3: 37,
}

check_114(
    channel_distribution_114.to_dict()
    == expected_distribution_114,
    "Single-, two-, and three-channel days match approval"
)

check_114(
    ground_truth_user_days_11[
        "is_active_day"
    ].eq(True).all()
    and ground_truth_user_days_11[
        "is_forced_offday"
    ].eq(False).all(),
    "Only active attack days are included"
)


# ------------------------------------------------------------
# STEP 9 — Verify daily rotation and synchronization
#
# These are execution-approved scheduling annotations,
# not behavioral features calculated for the model.
# ------------------------------------------------------------

schedule_annotations_114 = [
    "scenario_option_id",
    "primary_channel",
    "scheduled_channel_set",
    "is_switch",
    "sync_type",
]

for field in schedule_annotations_114:

    reference_114 = (
        channel_114
        .groupby(["user", "day"])[field]
        .agg(
            lambda values: (
                values.iloc[0]
                if values.nunique(dropna=False) == 1
                else "__INCONSISTENT__"
            )
        )
        .rename(f"reference__{field}")
        .reset_index()
    )

    comparison_114 = ground_truth_user_days_11[
        ["user", "day", field]
    ].merge(
        reference_114,
        on=["user", "day"],
        how="left",
        validate="one_to_one",
    )

    expected_114 = comparison_114[
        f"reference__{field}"
    ]

    actual_114 = comparison_114[field]

    matches_114 = (
        actual_114.astype("string")
        .eq(expected_114.astype("string"))
        .fillna(False)
        |
        (actual_114.isna() & expected_114.isna())
    )

    check_114(
        matches_114.all()
        and expected_114.astype("string")
            .ne("__INCONSISTENT__").all(),
        f"Execution evidence matches daily {field}"
    )


# ------------------------------------------------------------
# STEP 10 — Population, identity and event reconciliation
# ------------------------------------------------------------

check_114(
    len(ground_truth_user_days_11) == 9_080
    and ground_truth_user_days_11["user"].nunique() == 400,
    "Final User-Day Ground Truth population is exact"
)

check_114(
    ground_truth_user_days_11[
        ["user", "day"]
    ].notna().all().all()
    and not ground_truth_user_days_11.duplicated(
        ["user", "day"]
    ).any(),
    "Final User-Day keys are unique and non-null"
)

check_114(
    set(ground_truth_user_days_11["user"])
    == set(assignment_114["user"]),
    "Ground Truth users exactly match approved attackers"
)

check_114(
    int(
        ground_truth_user_days_11[
            "total_injected_events"
        ].sum()
    ) == 29_007,
    "Final User-Day totals equal 29,007 injected events"
)

check_114(
    int(
        ground_truth_user_days_11[
            "executed_channel_count"
        ].sum()
    ) == len(channel_114) == 10_124,
    "User-Day channel counts reconcile with Channel-Day GT"
)

check_114(
    "y" not in ground_truth_user_days_11.columns,
    "No model classification target y is included"
)


# ------------------------------------------------------------
# STEP 11 — Final approval and concise result
# ------------------------------------------------------------

user_day_ground_truth_qa_11 = pd.DataFrame(
    qa_rows_114
)

require_114(
    user_day_ground_truth_qa_11["status"]
    .eq("PASS").all(),
    "User-Day Ground Truth validation is incomplete"
)

user_day_ground_truth_approved_11 = True

distribution_summary_114 = pd.DataFrame({
    "executed_channels": [1, 2, 3],
    "attack_user_days": [
        int(channel_distribution_114.loc[i])
        for i in [1, 2, 3]
    ],
})

print("\n11.4 — DAILY CHANNEL DISTRIBUTION")
display(distribution_summary_114)

print("\n" + "=" * 58)
print("11.4 — USER-DAY GROUND TRUTH VALIDATION")
print("=" * 58)
print("Approved attackers       :",
      ground_truth_user_days_11["user"].nunique())
print("Attacked User-Days       :",
      len(ground_truth_user_days_11))
print("Executed channel-actions:",
      int(
          ground_truth_user_days_11[
              "executed_channel_count"
          ].sum()
      ))
print("Injected attack events  :",
      int(
          ground_truth_user_days_11[
              "total_injected_events"
          ].sum()
      ))
print("Ground Truth columns     :",
      len(ground_truth_user_days_11.columns))
print("Assignment fields retained:",
      len(assignment_114.columns))
print("QA checks                :",
      f"{len(user_day_ground_truth_qa_11)}/"
      f"{len(user_day_ground_truth_qa_11)} PASS")
print("Classification y added   : NO")
print("Source files modified    : NO")
print("Files exported           : NO")
print("Status                   : APPROVED")
print("=" * 58)



11.4 — DAILY CHANNEL DISTRIBUTION


,executed_channels,attack_user_days
0,1,8073
1,2,970
2,3,37



11.4 — USER-DAY GROUND TRUTH VALIDATION
Approved attackers       : 400
Attacked User-Days       : 9080
Executed channel-actions: 10124
Injected attack events  : 29007
Ground Truth columns     : 42
Assignment fields retained: 23
QA checks                : 27/27 PASS
Classification y added   : NO
Source files modified    : NO
Files exported           : NO
Status                   : APPROVED



## 11.5 — Independent Cross-Level Ground Truth Validation

### Objective

Independently validate the consistency, completeness, and accuracy of the two attack-only Ground Truth datasets before final export.

### Validation Scope

1. Verify the approved population, unique keys, and expected record counts.
2. Independently reconstruct daily injection totals and channel masks from the original execution log.
3. Verify the exact equivalence of Channel-Day Ground Truth and execution evidence.
4. Verify that all User-Day Ground Truth records correspond to executed attack days.
5. Verify per-channel and total event increments across both Ground Truth levels.
6. Reconcile scenario identifiers, approved assignment dimensions, switching indicators, and synchronization types.
7. Verify the final injected User-Day count fields against the actual execution records.
8. Confirm the exclusion of benign users, inactive days, and forced off-days.
9. Confirm that neither Ground Truth dataset contains the model target `y`.
10. Block export if any independent check fails.

### Expected Outcome

Two independently verified attack-only Ground Truth datasets, approved for protected export in Section 11.6.

This section is read-only and does not modify any source or Ground Truth files.


In [5]:

# ============================================================
# NOTEBOOK 11 — SECTION 11.5
# INDEPENDENT CROSS-LEVEL GROUND TRUTH VALIDATION
# READ-ONLY — NO EXPORTS
# ============================================================

import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal
from IPython.display import display


# ------------------------------------------------------------
# STEP 1 — Prerequisites
# ------------------------------------------------------------

def require_115(condition, message):
    if not bool(condition):
        raise AssertionError(f"11.5 FAILED: {message}")


require_115(
    channel_ground_truth_approved_11 is True
    and user_day_ground_truth_approved_11 is True
    and scenario_execution_approved_11 is True,
    "Sections 11.2, 11.3 and 11.4 must be approved."
)

qa_rows_115 = []


def check_115(condition, description):
    require_115(condition, description)
    qa_rows_115.append({
        "check": description,
        "status": "PASS",
    })


def normalize_keys_115(frame):
    result = frame.copy()
    result["user"] = result["user"].astype("string")
    result["day"] = pd.to_datetime(
        result["day"]
    ).dt.normalize()
    return result


def same_values_115(left, right):
    """Exact value comparison, treating paired nulls equally."""
    return bool(
        (
            left.reset_index(drop=True)
            .eq(right.reset_index(drop=True))
            .fillna(False)
            |
            (
                left.reset_index(drop=True).isna()
                &
                right.reset_index(drop=True).isna()
            )
        ).all()
    )


CHANNEL_BITS_115 = {
    "EMAIL": 1,
    "USB": 2,
    "HTTP": 4,
}


# ------------------------------------------------------------
# STEP 2 — Prepare independent source and output copies
# ------------------------------------------------------------

channel_gt_115 = normalize_keys_115(
    ground_truth_channel_days_11
)

user_day_gt_115 = normalize_keys_115(
    ground_truth_user_days_11
)

execution_115 = normalize_keys_115(
    injection_execution_log_11
)

schedule_115 = normalize_keys_115(
    daily_schedule_11
)

assignment_115 = approved_assignments_11.copy()
assignment_115["user"] = (
    assignment_115["user"].astype("string")
)

final_user_day_115 = normalize_keys_115(
    injected_user_day_11
)

check_115(
    len(channel_gt_115) == 10_124
    and len(user_day_gt_115) == 9_080
    and len(execution_115) == 10_124,
    "Expected Ground Truth and execution populations"
)

check_115(
    len(assignment_115) == 400
    and assignment_115["user"].nunique() == 400
    and channel_gt_115["user"].nunique() == 400
    and user_day_gt_115["user"].nunique() == 400,
    "Exactly 400 approved attackers at both levels"
)

check_115(
    not channel_gt_115.duplicated(
        ["user", "day", "channel"]
    ).any()
    and channel_gt_115[
        ["user", "day", "channel"]
    ].notna().all().all(),
    "Channel-Day Ground Truth keys are valid and unique"
)

check_115(
    not user_day_gt_115.duplicated(
        ["user", "day"]
    ).any()
    and user_day_gt_115[
        ["user", "day"]
    ].notna().all().all(),
    "User-Day Ground Truth keys are valid and unique"
)


# ------------------------------------------------------------
# STEP 3 — Independently verify Channel-Day GT
# against the actual Notebook 10 execution log
# ------------------------------------------------------------

execution_cols_115 = execution_115.columns.tolist()

source_sorted_115 = (
    execution_115
    .sort_values(["user", "day", "channel"])
    .reset_index(drop=True)
)

channel_sorted_115 = (
    channel_gt_115[execution_cols_115]
    .sort_values(["user", "day", "channel"])
    .reset_index(drop=True)
)

assert_frame_equal(
    source_sorted_115,
    channel_sorted_115,
    check_dtype=False,
    check_exact=True,
)

check_115(
    True,
    "All original execution-log values match Channel-Day GT"
)

check_115(
    (
        channel_gt_115["count_before"]
        + channel_gt_115["count_added"]
        == channel_gt_115["count_after"]
    ).all()
    and channel_gt_115["count_added"].gt(0).all(),
    "Channel-level before/add/after values are consistent"
)


# ------------------------------------------------------------
# STEP 4 — Independently reconstruct User-Day evidence
# from the original execution log
# ------------------------------------------------------------

execution_115["independent_channel_bit"] = (
    execution_115["channel"]
    .map(CHANNEL_BITS_115)
)

check_115(
    execution_115["independent_channel_bit"]
    .notna().all(),
    "All executed channels have valid channel bits"
)

independent_daily_115 = (
    execution_115
    .groupby(["user", "day"], as_index=False)
    .agg(
        expected_channel_count=(
            "channel",
            "size"
        ),
        expected_total_events=(
            "count_added",
            "sum"
        ),
        expected_channel_mask=(
            "independent_channel_bit",
            lambda values: int(
                np.bitwise_or.reduce(
                    values.to_numpy(dtype=np.int64)
                )
            )
        ),
    )
)

independent_channels_115 = (
    execution_115
    .pivot(
        index=["user", "day"],
        columns="channel",
        values="count_added",
    )
    .reindex(columns=["EMAIL", "USB", "HTTP"])
    .fillna(0)
    .astype("int64")
    .rename(columns={
        "EMAIL": "expected_email",
        "USB": "expected_usb",
        "HTTP": "expected_http",
    })
    .reset_index()
)

independent_daily_115 = independent_daily_115.merge(
    independent_channels_115,
    on=["user", "day"],
    how="left",
    validate="one_to_one",
)

check_115(
    len(independent_daily_115) == 9_080
    and not independent_daily_115.duplicated(
        ["user", "day"]
    ).any(),
    "Independent reconstruction has exactly 9,080 unique days"
)


# ------------------------------------------------------------
# STEP 5 — Match every reconstructed attack day
# ------------------------------------------------------------

daily_compare_115 = user_day_gt_115.merge(
    independent_daily_115,
    on=["user", "day"],
    how="outer",
    validate="one_to_one",
    indicator=True,
)

check_115(
    len(daily_compare_115) == 9_080
    and daily_compare_115["_merge"].eq("both").all(),
    "User-Day Ground Truth matches every executed attack day"
)

comparison_pairs_115 = {
    "executed_channel_count": "expected_channel_count",
    "total_injected_events": "expected_total_events",
    "executed_channel_mask": "expected_channel_mask",
    "email_injected_events": "expected_email",
    "usb_injected_events": "expected_usb",
    "http_injected_events": "expected_http",
}

for actual, expected in comparison_pairs_115.items():
    check_115(
        same_values_115(
            daily_compare_115[actual],
            daily_compare_115[expected],
        ),
        f"Exact independent User-Day reconciliation: {actual}"
    )


# ------------------------------------------------------------
# STEP 6 — Verify attack-day channel distribution
# ------------------------------------------------------------

distribution_115 = (
    independent_daily_115["expected_channel_count"]
    .value_counts()
    .to_dict()
)

check_115(
    distribution_115 == {
        1: 8073,
        2: 970,
        3: 37,
    },
    "Independent 1/2/3-channel day distribution"
)

check_115(
    int(
        independent_daily_115[
            "expected_channel_count"
        ].sum()
    ) == 10_124,
    "Daily channel counts sum to 10,124 actions"
)

check_115(
    int(
        independent_daily_115[
            "expected_total_events"
        ].sum()
    ) == 29_007,
    "Independent daily totals sum to 29,007 events"
)


# ------------------------------------------------------------
# STEP 7 — Verify approved daily schedule annotations
# ------------------------------------------------------------

active_schedule_115 = schedule_115.loc[
    schedule_115["is_active_day"].eq(True)
].copy()

check_115(
    len(active_schedule_115) == 9_080,
    "Approved schedule contains exactly 9,080 active days"
)

schedule_compare_fields_115 = [
    "scenario_option_id",
    "window_workdays",
    "window_start",
    "window_end",
    "channel_configuration",
    "switching_level",
    "stealth_level",
    "stealth_multiplier",
    "forced_offdays",
    "workday_index",
    "active_day_index",
    "primary_channel",
    "primary_channel_mask",
    "scheduled_channel_set",
    "scheduled_channel_mask",
    "scheduled_channel_count",
    "is_switch",
    "sync_type",
]

schedule_reference_115 = active_schedule_115[
    ["user", "day", *schedule_compare_fields_115]
].rename(columns={
    col: f"reference__{col}"
    for col in schedule_compare_fields_115
})

schedule_comparison_115 = user_day_gt_115.merge(
    schedule_reference_115,
    on=["user", "day"],
    how="outer",
    validate="one_to_one",
    indicator=True,
)

check_115(
    schedule_comparison_115["_merge"].eq("both").all()
    and len(schedule_comparison_115) == 9_080,
    "All User-Day GT records match active scheduled days"
)

for field in schedule_compare_fields_115:

    actual = schedule_comparison_115[field]
    expected = schedule_comparison_115[
        f"reference__{field}"
    ]

    if field in ("window_start", "window_end"):
        actual = pd.to_datetime(actual)
        expected = pd.to_datetime(expected)

    check_115(
        same_values_115(actual, expected),
        f"Daily schedule annotation matches: {field}"
    )


# ------------------------------------------------------------
# STEP 8 — Verify approved scenario assignments
# ------------------------------------------------------------

check_115(
    set(user_day_gt_115["user"])
    == set(assignment_115["user"]),
    "Ground Truth users match all approved assignments"
)

assignment_fields_115 = [
    "scenario_option_id",
    "channel_configuration",
    "switching_level",
    "stealth_level",
    "stealth_multiplier",
    "window_workdays",
    "window_start",
    "window_end",
]

assignment_reference_115 = assignment_115[
    ["user", *assignment_fields_115]
].rename(columns={
    col: f"reference__{col}"
    for col in assignment_fields_115
})

assignment_comparison_115 = user_day_gt_115.merge(
    assignment_reference_115,
    on="user",
    how="left",
    validate="many_to_one",
    indicator=True,
)

check_115(
    assignment_comparison_115["_merge"].eq("both").all(),
    "Every attack day has an approved attacker assignment"
)

for field in assignment_fields_115:

    actual = assignment_comparison_115[field]
    expected = assignment_comparison_115[
        f"reference__{field}"
    ]

    if field in ("window_start", "window_end"):
        actual = pd.to_datetime(actual)
        expected = pd.to_datetime(expected)

    check_115(
        same_values_115(actual, expected),
        f"Approved assignment matches: {field}"
    )

check_115(
    set(assignment_115.columns).issubset(
        user_day_gt_115.columns
    ),
    "All 23 approved assignment fields are preserved"
)


# ------------------------------------------------------------
# STEP 9 — Independently verify actual injected User-Day
# counts against execution evidence
# ------------------------------------------------------------

COUNT_FIELD_BY_CHANNEL_115 = {
    "EMAIL": "external_email_count",
    "USB": "usb_copy_count",
    "HTTP": "www_upload_count",
}

expected_count_fields_115 = (
    execution_115["channel"]
    .map(COUNT_FIELD_BY_CHANNEL_115)
)

check_115(
    same_values_115(
        execution_115["source_count_field"],
        expected_count_fields_115,
    ),
    "Execution records reference the correct count fields"
)

actual_counts_115 = final_user_day_115[
    [
        "user",
        "day",
        "external_email_count",
        "usb_copy_count",
        "www_upload_count",
    ]
].melt(
    id_vars=["user", "day"],
    var_name="source_count_field",
    value_name="actual_final_count",
)

actual_final_comparison_115 = execution_115[
    [
        "user",
        "day",
        "source_count_field",
        "count_after",
    ]
].merge(
    actual_counts_115,
    on=["user", "day", "source_count_field"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

check_115(
    actual_final_comparison_115["_merge"]
    .eq("both").all()
    and len(actual_final_comparison_115) == 10_124,
    "Every execution action maps to the injected User-Day"
)

check_115(
    same_values_115(
        actual_final_comparison_115["count_after"],
        actual_final_comparison_115["actual_final_count"],
    ),
    "All final injected channel counts match execution evidence"
)

del actual_counts_115


# ------------------------------------------------------------
# STEP 10 — Exclusion and label-leakage checks
# ------------------------------------------------------------

offdays_115 = schedule_115.loc[
    schedule_115["is_forced_offday"].eq(True),
    ["user", "day"],
]

offday_overlap_115 = user_day_gt_115[
    ["user", "day"]
].merge(
    offdays_115,
    on=["user", "day"],
    how="inner",
)

check_115(
    len(offdays_115) == 1_670
    and offday_overlap_115.empty,
    "All 1,670 forced off-days are excluded"
)

check_115(
    user_day_gt_115["is_active_day"].eq(True).all()
    and user_day_gt_115["is_forced_offday"].eq(False).all(),
    "Only actually executed active attack days are included"
)

check_115(
    "y" not in channel_gt_115.columns
    and "y" not in user_day_gt_115.columns,
    "Both Ground Truth datasets exclude classification target y"
)

check_115(
    len(final_user_day_115) == 1_394_010
    and final_user_day_115["user"].nunique() == 4000,
    "Final injected User-Day retains the complete population"
)


# ------------------------------------------------------------
# STEP 11 — Independent cross-level reconciliation
# ------------------------------------------------------------

channel_totals_115 = (
    execution_115
    .groupby("channel", as_index=False)
    .agg(
        channel_actions=("user", "size"),
        injected_events=("count_added", "sum"),
    )
)

gt_channel_totals_115 = (
    channel_gt_115
    .groupby("channel", as_index=False)
    .agg(
        gt_actions=("user", "size"),
        gt_events=("count_added", "sum"),
    )
)

channel_reconciliation_115 = channel_totals_115.merge(
    gt_channel_totals_115,
    on="channel",
    validate="one_to_one",
)

check_115(
    (
        channel_reconciliation_115["channel_actions"]
        == channel_reconciliation_115["gt_actions"]
    ).all()
    and (
        channel_reconciliation_115["injected_events"]
        == channel_reconciliation_115["gt_events"]
    ).all(),
    "Channel-level execution and Ground Truth totals match"
)

check_115(
    int(user_day_gt_115["total_injected_events"].sum())
    == int(channel_gt_115["count_added"].sum())
    == 29_007,
    "Both Ground Truth levels reconcile to 29,007 events"
)


# ------------------------------------------------------------
# STEP 12 — Approval and concise QA summary
# ------------------------------------------------------------

independent_ground_truth_qa_11 = pd.DataFrame(
    qa_rows_115
)

require_115(
    independent_ground_truth_qa_11["status"]
    .eq("PASS").all(),
    "Independent Ground Truth validation is incomplete"
)

independent_ground_truth_approved_11 = True

print("\n11.5 — INDEPENDENT CHANNEL RECONCILIATION")
display(channel_reconciliation_115)

print("\n" + "=" * 60)
print("11.5 — INDEPENDENT GROUND TRUTH VALIDATION")
print("=" * 60)
print("Approved attackers       : 400")
print("Channel-Day GT records   :", len(channel_gt_115))
print("User-Day GT records      :", len(user_day_gt_115))
print("Injected attack events   :",
      int(channel_gt_115["count_added"].sum()))
print("User-Day GT columns      :", len(user_day_gt_115.columns))
print("Channel-Day GT columns   :", len(channel_gt_115.columns))
print(
    "Independent QA           :",
    f"{len(independent_ground_truth_qa_11)}/"
    f"{len(independent_ground_truth_qa_11)} PASS"
)
print("Scenario reconciliation  : PASS")
print("Executed channel counts  : EXACT MATCH")
print("Forced off-days excluded : YES")
print("Classification y added   : NO")
print("Files exported           : NO")
print("FINAL EXPORT APPROVAL    : APPROVED")
print("=" * 60)



11.5 — INDEPENDENT CHANNEL RECONCILIATION


,channel,channel_actions,injected_events,gt_actions,gt_events
0,EMAIL,6703,17791,6703,17791
1,HTTP,1486,6296,1486,6296
2,USB,1935,4920,1935,4920



11.5 — INDEPENDENT GROUND TRUTH VALIDATION
Approved attackers       : 400
Channel-Day GT records   : 10124
User-Day GT records      : 9080
Injected attack events   : 29007
User-Day GT columns      : 42
Channel-Day GT columns   : 27
Independent QA           : 58/58 PASS
Scenario reconciliation  : PASS
Executed channel counts  : EXACT MATCH
Forced off-days excluded : YES
Classification y added   : NO
Files exported           : NO
FINAL EXPORT APPROVAL    : APPROVED


In [6]:

from IPython.display import display
import pandas as pd

for name, df in {
    "User-Day Ground Truth": ground_truth_user_days_11,
    "Channel-Day Ground Truth": ground_truth_channel_days_11,
}.items():
    print(f"\n{name} — {len(df):,} rows × {len(df.columns)} columns")

    display(pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
    }))



User-Day Ground Truth — 9,080 rows × 42 columns


,column,dtype
0,user,string
1,day,datetime64[ms]
2,scenario_option_id,uint32
3,window_workdays,int64
4,window_start,datetime64[ms]
5,window_end,datetime64[ms]
6,channel_configuration,category
7,switching_level,category
8,stealth_level,category
9,stealth_multiplier,float64



Channel-Day Ground Truth — 10,124 rows × 27 columns


,column,dtype
0,user,string
1,day,datetime64[ms]
2,scenario_option_id,uint32
3,workday_index,int64
4,channel,str
5,source_count_field,str
6,channel_configuration,category
7,primary_channel,str
8,scheduled_channel_set,str
9,sync_type,str


In [7]:

user_cols = set(ground_truth_user_days_11.columns)
channel_cols = set(ground_truth_channel_days_11.columns)

shared = sorted(user_cols & channel_cols)
user_only = sorted(user_cols - channel_cols)
channel_only = sorted(channel_cols - user_cols)

print(f"Shared columns       : {len(shared)}")
print(f"User-Day only        : {len(user_only)}")
print(f"Channel-Day only     : {len(channel_only)}")
print(f"Total unique columns: {len(user_cols | channel_cols)}")

print("\nSHARED:", shared)
print("\nUSER-DAY ONLY:", user_only)
print("\nCHANNEL-DAY ONLY:", channel_only)


Shared columns       : 21
User-Day only        : 21
Channel-Day only     : 6
Total unique columns: 48

SHARED: ['active_day_index', 'channel_configuration', 'day', 'forced_offdays', 'formula_id', 'is_active_day', 'is_forced_offday', 'is_switch', 'primary_channel', 'scenario_option_id', 'scheduled_channel_mask', 'scheduled_channel_set', 'stealth_level', 'stealth_multiplier', 'switching_level', 'sync_type', 'user', 'window_end', 'window_start', 'window_workdays', 'workday_index']

USER-DAY ONLY: ['all_selected_channels_have_reference', 'assignment_status', 'catalog_status', 'channel_count', 'email_injected_events', 'executed_channel_count', 'executed_channel_mask', 'http_injected_events', 'injected_channels', 'max_abs_recurrence_gap_pp', 'max_positive_recurrence_increase_pp', 'multiplier_configuration', 'primary_channel_mask', 'proof_source', 'proposed_active_days', 'scenario_type_id', 'scheduled_channel_count', 'set_switch_bitmask', 'source_validation_status', 'total_injected_events', '


## 11.6 — Final Export & Integrity Verification

### Objective

Safely export the two independently validated Ground Truth datasets and their supporting documentation, while preserving source integrity and complete traceability.

### Main Ground Truth Deliverables

1. **`ground_truth_user_days_r62.parquet`**
   - Granularity: One record per attacked User-Day.
   - Expected shape: 9,080 rows × 42 columns.
   - Includes approved scenario dimensions, attack timelines, channel rotation, synchronization, and executed daily injection summaries.

2. **`ground_truth_channel_days_r62.parquet`**
   - Granularity: One record per executed User-Day-Channel.
   - Expected shape: 10,124 rows × 27 columns.
   - Includes channel-level injection evidence, scheduling context, and scenario information.

### Supporting Deliverables

- `ground_truth_data_dictionary.csv`: Column names, types, nullability, descriptions, and provenance.
- Four QA reports covering Sections 11.2–11.5.
- `notebook_summary.md`: Final methodology and validation summary.
- `export_manifest.json`: Artifact metadata and SHA-256 checksums.

### Export and Integrity Controls

1. Require all preceding Notebook 11 validations to pass.
2. Verify Ground Truth dimensions, keys, populations, and injected-event totals.
3. Preserve the approved source datasets without modification.
4. Stage all outputs before publication.
5. Verify exact Parquet read-back equality.
6. Verify supporting files and their SHA-256 checksums.
7. Prevent overwriting existing destination files.
8. Roll back newly published artifacts if publication or final verification fails.
9. Approve the export only after all integrity controls pass.

### Restrictions

- No additional injection or modification of User-Day records.
- No historical baseline recalculation.
- No model feature engineering.
- No classification target `y`.
- No separate User-Level Ground Truth file.

### Output Directory

`New_Attack_Simulation/reports/11_Ground_Truth_and_Quality_Validation/`

### Expected Outcome

Two validated, immutable-in-content Ground Truth exports, accompanied by documented schemas, quality reports, and reproducible integrity evidence, ready for Notebook 12.


In [8]:

# ============================================================
# NOTEBOOK 11 — SECTION 11.6
# FINAL EXPORT & INTEGRITY VERIFICATION
#
# 2 Ground Truth datasets + 7 supporting files
# Protected publication | Exact read-back | SHA-256
# No overwrite | No source modifications
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil
import tempfile

import pandas as pd
from pandas.testing import assert_frame_equal
from IPython.display import display


# ------------------------------------------------------------
# STEP 1 — Enforce approved validation prerequisites
# ------------------------------------------------------------

def require_116(condition, message):
    if not bool(condition):
        raise RuntimeError(f"11.6 EXPORT BLOCKED: {message}")


approval_flags_116 = {
    "11.2": scenario_execution_approved_11,
    "11.3": channel_ground_truth_approved_11,
    "11.4": user_day_ground_truth_approved_11,
    "11.5": independent_ground_truth_approved_11,
}

qa_reports_116 = {
    "scenario_execution_qa.csv": (
        scenario_execution_qa_11, 35
    ),
    "channel_ground_truth_qa.csv": (
        channel_ground_truth_qa_11, 20
    ),
    "user_day_ground_truth_qa.csv": (
        user_day_ground_truth_qa_11, 27
    ),
    "independent_ground_truth_qa.csv": (
        independent_ground_truth_qa_11, 58
    ),
}

for section, approved in approval_flags_116.items():
    require_116(
        approved is True,
        f"Section {section} has not been approved."
    )

for filename, (report, expected_checks) in qa_reports_116.items():
    require_116(
        len(report) == expected_checks
        and {"check", "status"}.issubset(report.columns)
        and report["status"].eq("PASS").all(),
        f"Validation report failed: {filename}"
    )


# ------------------------------------------------------------
# STEP 2 — Prepare and validate final Ground Truth datasets
# ------------------------------------------------------------

GT_EXPORTS_116 = {
    "ground_truth_user_days_r62.parquet":
        ground_truth_user_days_11,

    "ground_truth_channel_days_r62.parquet":
        ground_truth_channel_days_11,
}

user_gt_116 = ground_truth_user_days_11
channel_gt_116 = ground_truth_channel_days_11

require_116(
    user_gt_116.shape == (9_080, 42),
    "User-Day GT dimensions differ from approval."
)

require_116(
    channel_gt_116.shape == (10_124, 27),
    "Channel-Day GT dimensions differ from approval."
)

for name, frame, keys in (
    ("User-Day", user_gt_116, ["user", "day"]),
    (
        "Channel-Day",
        channel_gt_116,
        ["user", "day", "channel"],
    ),
):
    require_116(
        frame[keys].notna().all().all()
        and not frame.duplicated(keys).any(),
        f"{name} GT has invalid or duplicate keys."
    )

    require_116(
        frame["user"].nunique() == 400,
        f"{name} GT must contain exactly 400 attackers."
    )

    require_116(
        "y" not in frame.columns,
        f"{name} GT must not contain the model target y."
    )

require_116(
    int(user_gt_116["total_injected_events"].sum())
    == int(channel_gt_116["count_added"].sum())
    == 29_007,
    "Ground Truth injected-event totals mismatch."
)

require_116(
    int(user_gt_116["executed_channel_count"].sum())
    == len(channel_gt_116)
    == 10_124,
    "Cross-level channel-action counts mismatch."
)

require_116(
    set(approved_assignments_11.columns).issubset(
        user_gt_116.columns
    ),
    "Approved scenario dimensions are missing."
)


# ------------------------------------------------------------
# STEP 3 — Protect and fingerprint all input sources
# ------------------------------------------------------------

def sha256_116(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as stream:
        for chunk in iter(
            lambda: stream.read(8 * 1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


source_paths_116 = {
    "notebook08_assignments": Path(
        assignment_source_112
    ),
    "notebook09_schedule": Path(
        schedule_path_11
    ),
    "notebook10_user_day": (
        REPORT_10_11
        / "user_day_injected_counts_r62.parquet"
    ),
    "notebook10_execution_log": (
        REPORT_10_11
        / "injection_execution_log_r62.parquet"
    ),
    "notebook10_manifest": (
        REPORT_10_11 / "export_manifest.json"
    ),
}

for name, path in source_paths_116.items():
    require_116(
        path.is_file(),
        f"Required source file missing: {name}"
    )

source_hashes_116 = {
    name: sha256_116(path)
    for name, path in source_paths_116.items()
}

# Confirm Notebook 10's original manifest still agrees
# with the two directly consumed execution artifacts.
for filename in (
    "user_day_injected_counts_r62.parquet",
    "injection_execution_log_r62.parquet",
):
    path = REPORT_10_11 / filename

    require_116(
        sha256_116(path)
        == manifest_11["artifacts"][filename]["sha256"],
        f"Notebook 10 source checksum mismatch: {filename}"
    )


# ------------------------------------------------------------
# STEP 4 — Define final export destination
# ------------------------------------------------------------

REPORT_11 = (
    ROOT_11
    / "reports"
    / "11_Ground_Truth_and_Quality_Validation"
)

REPORT_11.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# STEP 5 — Generate a dictionary from ACTUAL GT schemas
#
# Definitions are explicit for established fields.
# Unknown Notebook 08 metadata is identified as imported,
# rather than assigning an unsupported interpretation.
# ------------------------------------------------------------

descriptions_116 = {
    "user": "Unique user identifier.",
    "day": "Calendar date of the executed attack.",

    "channel": (
        "Channel on which the injection was executed: "
        "EMAIL, USB, or HTTP."
    ),
    "scenario_option_id": (
        "Identifier of the approved attacker scenario."
    ),
    "channel_configuration": (
        "Approved channel configuration assigned to the attacker."
    ),
    "switching_level": (
        "Approved scenario-level channel-switching setting."
    ),
    "stealth_level": (
        "Approved stealth category used for count injection."
    ),
    "stealth_multiplier": (
        "Multiplier applied to the historical P95 "
        "in injection Formula A."
    ),
    "formula_id": (
        "Identifier of the approved count-injection formula."
    ),

    "window_workdays": (
        "Number of scheduled workdays in the attack window."
    ),
    "window_start": "Start date of the approved attack window.",
    "window_end": "End date of the approved attack window.",
    "workday_index": (
        "Position of the date within the scheduled workday sequence."
    ),
    "active_day_index": (
        "Position of the date within active injection days."
    ),
    "forced_offdays": (
        "Approved count of forced off-days in the attack window."
    ),
    "is_active_day": (
        "Whether the scheduled date is an active injection day."
    ),
    "is_forced_offday": (
        "Whether the scheduled date is a forced off-day."
    ),

    "primary_channel": (
        "Primary channel selected by the approved daily schedule."
    ),
    "primary_channel_mask": (
        "Bitmask representing the scheduled primary channel."
    ),
    "scheduled_channel_set": (
        "Channels scheduled for injection on the date."
    ),
    "scheduled_channel_mask": (
        "Bitmask of scheduled injection channels; "
        "EMAIL=1, USB=2, HTTP=4."
    ),
    "scheduled_channel_count": (
        "Number of channels scheduled for the attack day."
    ),
    "is_switch": (
        "Approved schedule indicator of a primary-channel switch."
    ),
    "sync_type": (
        "Approved same-day channel synchronization category."
    ),

    "source_count_field": (
        "User-Day event-count column targeted by the injection."
    ),
    "baseline_p95": (
        "Precomputed historical user-channel P95 "
        "used by Formula A."
    ),
    "count_before": (
        "Observed channel event count before injection."
    ),
    "count_added": (
        "Number of attack events added to the channel."
    ),
    "count_after": (
        "Channel event count after injection."
    ),

    "executed_channel_count": (
        "Number of channels actually injected on the attack day."
    ),
    "executed_channel_mask": (
        "Bitmask of actually injected channels; "
        "EMAIL=1, USB=2, HTTP=4."
    ),
    "injected_channels": (
        "Readable set of channels actually injected on the date."
    ),
    "total_injected_events": (
        "Sum of attack-event increments across all channels "
        "on the User-Day."
    ),
    "email_injected_events": (
        "Attack-event increments injected via EMAIL on the date."
    ),
    "usb_injected_events": (
        "Attack-event increments injected via USB on the date."
    ),
    "http_injected_events": (
        "Attack-event increments injected via HTTP on the date."
    ),
}

execution_columns_116 = set(
    injection_execution_log_11.columns
)
schedule_columns_116 = set(daily_schedule_11.columns)
assignment_columns_116 = set(
    approved_assignments_11.columns
)


def field_origin_116(column):
    sources = []

    if column in assignment_columns_116:
        sources.append("Notebook 08 — Approved Assignments")

    if column in schedule_columns_116:
        sources.append("Notebook 09 — Daily Schedule")

    if column in execution_columns_116:
        sources.append("Notebook 10 — Execution Log")

    if column in {
        "executed_channel_count",
        "executed_channel_mask",
        "injected_channels",
        "total_injected_events",
        "email_injected_events",
        "usb_injected_events",
        "http_injected_events",
    }:
        sources.append("Notebook 11 — Derived from Execution")

    return "; ".join(sources) or "Notebook 11"


dictionary_rows_116 = []

for filename, frame in GT_EXPORTS_116.items():

    for position, column in enumerate(frame.columns, start=1):

        # The dictionary must describe what is known.
        # Unrecognized approved-assignment fields remain
        # traceable without inventing their semantics.
        description = descriptions_116.get(column)

        if description is None:
            require_116(
                column in assignment_columns_116,
                f"Unrecognized Ground Truth column: {column}"
            )

            description = (
                "Additional approved scenario-assignment "
                "attribute imported from Notebook 08; "
                "refer to its assignment definition."
            )

        dictionary_rows_116.append({
            "dataset": filename,
            "position": position,
            "column": column,
            "dtype": str(frame[column].dtype),
            "nullable": bool(frame[column].isna().any()),
            "description": description,
            "provenance": field_origin_116(column),
        })

ground_truth_dictionary_11 = pd.DataFrame(
    dictionary_rows_116
)

require_116(
    len(ground_truth_dictionary_11) == 42 + 27
    and ground_truth_dictionary_11[
        ["dataset", "column"]
    ].duplicated().sum() == 0,
    "Data Dictionary does not match both GT schemas."
)


# ------------------------------------------------------------
# STEP 6 — Prepare QA reports and notebook summary
# ------------------------------------------------------------

CSV_EXPORTS_116 = {
    "ground_truth_data_dictionary.csv":
        ground_truth_dictionary_11,

    **{
        filename: report
        for filename, (report, _) in qa_reports_116.items()
    },
}

summary_116 = """# Notebook 11 — Ground Truth and Quality Validation

## Status

COMPLETE — INDEPENDENT VALIDATION APPROVED

## Ground Truth Datasets

### User-Day Ground Truth
- File: ground_truth_user_days_r62.parquet
- Records: 9,080
- Columns: 42
- Granularity: One executed attack User-Day

### Channel-Day Ground Truth
- File: ground_truth_channel_days_r62.parquet
- Records: 10,124
- Columns: 27
- Granularity: One executed User-Day-Channel action

## Approved Attack Population

- Attackers: 400
- Active attacked User-Days: 9,080
- Verified channel-actions: 10,124
- Additional injected attack events: 29,007
- Forced off-days excluded: 1,670

## Channel-Day Distribution

- EMAIL: 6,703 actions; 17,791 events
- USB: 1,935 actions; 4,920 events
- HTTP: 1,486 actions; 6,296 events

## User-Day Channel Distribution

- Single-channel days: 8,073
- Two-channel days: 970
- Three-channel days: 37

## QA Results

- Section 11.2: 35/35 PASS
- Section 11.3: 20/20 PASS
- Section 11.4: 27/27 PASS
- Section 11.5: 58/58 PASS

## Provenance

- Notebook 08: Approved attacker scenario assignments
- Notebook 09: Daily attack schedule
- Notebook 10: Verified injection execution evidence
- Notebook 11: Ground Truth construction and validation

## Data Boundaries

Both Ground Truth datasets contain executed attacks only.

No classification target y is included.

No User-Level Ground Truth is separately exported.

No behavioral features were calculated.

No source dataset was modified.

## Next Stage

Notebook 12 — Injected Feature Engineering and Validation.

The binary classification target y will be constructed
by matching Behavioral Features (user, day) against the
attack-only User-Day Ground Truth.
"""


# ------------------------------------------------------------
# STEP 7 — Reject all existing destination filenames
# ------------------------------------------------------------

ALL_NAMES_116 = (
    list(GT_EXPORTS_116)
    + list(CSV_EXPORTS_116)
    + ["notebook_summary.md", "export_manifest.json"]
)

require_116(
    len(ALL_NAMES_116) == len(set(ALL_NAMES_116)),
    "Duplicate export filenames."
)

existing_116 = [
    name
    for name in ALL_NAMES_116
    if (REPORT_11 / name).exists()
]

require_116(
    not existing_116,
    "Destination files already exist: "
    + ", ".join(existing_116)
)


# ------------------------------------------------------------
# STEP 8 — Exact Parquet round-trip helper
# ------------------------------------------------------------

def normalize_frame_116(frame):
    result = frame.reset_index(drop=True).copy()

    date_fields = {
        "day",
        "window_start",
        "window_end",
    }

    for column in result.columns:

        if column in date_fields:
            result[column] = pd.to_datetime(
                result[column]
            ).dt.normalize()

        elif isinstance(
            result[column].dtype,
            pd.CategoricalDtype,
        ):
            result[column] = (
                result[column].astype("string")
            )

        elif pd.api.types.is_string_dtype(
            result[column].dtype
        ):
            result[column] = (
                result[column].astype("string")
            )

    return result


def verify_parquet_116(path, original):
    restored = pd.read_parquet(path)

    require_116(
        restored.columns.tolist() == original.columns.tolist()
        and len(restored) == len(original),
        f"Parquet schema/row mismatch: {Path(path).name}"
    )

    assert_frame_equal(
        normalize_frame_116(original),
        normalize_frame_116(restored),
        check_dtype=False,
        check_exact=True,
        check_index_type=False,
        check_freq=False,
        check_categorical=False,
    )

    return restored


# ------------------------------------------------------------
# STEP 9 — Stage every artifact and verify before publication
# ------------------------------------------------------------

artifact_metadata_116 = {}
published_paths_116 = []
export_complete_116 = False

with tempfile.TemporaryDirectory(
    prefix=".staging_notebook11_",
    dir=REPORT_11,
) as temporary_directory:

    staging_116 = Path(temporary_directory)

    # 9.1 — Stage validated Ground Truth Parquet files.

    for filename, frame in GT_EXPORTS_116.items():

        staged = staging_116 / filename

        frame.to_parquet(
            staged,
            engine="pyarrow",
            index=False,
        )

        restored = verify_parquet_116(staged, frame)

        artifact_metadata_116[filename] = {
            "format": "parquet",
            "rows": int(len(restored)),
            "columns": int(len(restored.columns)),
            "size_bytes": staged.stat().st_size,
            "sha256": sha256_116(staged),
        }

        del restored

    # 9.2 — Stage supporting CSV reports.

    for filename, frame in CSV_EXPORTS_116.items():

        staged = staging_116 / filename

        csv_text = frame.to_csv(
            index=False,
            lineterminator="\n",
        )

        staged.write_text(
            csv_text,
            encoding="utf-8-sig",
        )

        require_116(
            staged.read_text(encoding="utf-8-sig")
            == csv_text,
            f"CSV read-back mismatch: {filename}"
        )

        restored = pd.read_csv(staged)

        require_116(
            len(restored) == len(frame)
            and restored.columns.tolist()
                == frame.columns.tolist(),
            f"CSV structure mismatch: {filename}"
        )

        artifact_metadata_116[filename] = {
            "format": "csv",
            "rows": int(len(frame)),
            "columns": int(len(frame.columns)),
            "size_bytes": staged.stat().st_size,
            "sha256": sha256_116(staged),
        }

        del restored

    # 9.3 — Stage Notebook 11 summary.

    summary_path = staging_116 / "notebook_summary.md"

    summary_path.write_text(
        summary_116,
        encoding="utf-8",
    )

    require_116(
        summary_path.read_text(encoding="utf-8")
        == summary_116,
        "Notebook summary read-back mismatch."
    )

    artifact_metadata_116["notebook_summary.md"] = {
        "format": "markdown",
        "size_bytes": summary_path.stat().st_size,
        "sha256": sha256_116(summary_path),
    }

    # 9.4 — Build the export manifest.

    export_manifest_11 = {
        "notebook": "11_Ground_Truth_and_Quality_Validation",
        "exported_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "status": "INDEPENDENT_VALIDATION_APPROVED",
        "ground_truth_datasets": {
            "user_day": {
                "filename":
                    "ground_truth_user_days_r62.parquet",
                "rows": 9080,
                "columns": 42,
            },
            "channel_day": {
                "filename":
                    "ground_truth_channel_days_r62.parquet",
                "rows": 10124,
                "columns": 27,
            },
        },
        "population": {
            "approved_attackers": 400,
            "attacked_user_days": 9080,
            "forced_offdays_excluded": 1670,
            "channel_actions": 10124,
            "injected_events": 29007,
        },
        "validation": {
            "scenario_execution": 35,
            "channel_ground_truth": 20,
            "user_day_ground_truth": 27,
            "independent_ground_truth": 58,
        },
        "contains_target_y": False,
        "contains_behavioral_features": False,
        "source_sha256": source_hashes_116,
        "artifacts": artifact_metadata_116,
    }

    manifest_file = staging_116 / "export_manifest.json"

    manifest_file.write_text(
        json.dumps(
            export_manifest_11,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    require_116(
        json.loads(
            manifest_file.read_text(encoding="utf-8")
        ) == export_manifest_11,
        "Staged export manifest mismatch."
    )

    # 9.5 — Verify ALL staged artifacts.

    for filename, metadata in artifact_metadata_116.items():

        staged = staging_116 / filename

        require_116(
            staged.stat().st_size == metadata["size_bytes"]
            and sha256_116(staged) == metadata["sha256"],
            f"Staged checksum mismatch: {filename}"
        )

    # 9.6 — Ensure original source files are unchanged.

    for name, path in source_paths_116.items():
        require_116(
            sha256_116(path) == source_hashes_116[name],
            f"Source file changed during staging: {name}"
        )

    # --------------------------------------------------------
    # STEP 10 — Protected publication
    #
    # Exclusive file creation prevents overwrites.
    # Failed publication rolls back only files created here.
    # --------------------------------------------------------

    try:

        for filename in ALL_NAMES_116:

            staged = staging_116 / filename
            destination = REPORT_11 / filename

            with staged.open("rb") as input_stream:
                with destination.open("xb") as output_stream:

                    published_paths_116.append(destination)

                    shutil.copyfileobj(
                        input_stream,
                        output_stream,
                        length=8 * 1024 * 1024,
                    )

            require_116(
                sha256_116(destination)
                == sha256_116(staged),
                f"Published checksum mismatch: {filename}"
            )

        # 10.1 — Verify final Ground Truth read-back.

        for filename, original in GT_EXPORTS_116.items():

            restored = verify_parquet_116(
                REPORT_11 / filename,
                original,
            )

            del restored

        # 10.2 — Verify all published artifacts.

        for filename, metadata in artifact_metadata_116.items():

            destination = REPORT_11 / filename

            require_116(
                destination.stat().st_size
                    == metadata["size_bytes"]
                and sha256_116(destination)
                    == metadata["sha256"],
                f"Final checksum mismatch: {filename}"
            )

        # 10.3 — Verify the final manifest.

        require_116(
            json.loads(
                (REPORT_11 / "export_manifest.json")
                .read_text(encoding="utf-8")
            ) == export_manifest_11,
            "Published manifest mismatch."
        )

        # 10.4 — Verify all original sources again.

        for name, path in source_paths_116.items():

            require_116(
                sha256_116(path)
                == source_hashes_116[name],
                f"Original source modified: {name}"
            )

        export_complete_116 = True

    except Exception:

        for path in reversed(published_paths_116):
            if path.exists():
                path.unlink()

        raise


# ------------------------------------------------------------
# STEP 11 — Final approval and concise result
# ------------------------------------------------------------

require_116(
    export_complete_116,
    "Protected publication did not complete."
)

ground_truth_export_inventory_11 = pd.DataFrame([
    {
        "filename": filename,
        "rows": details.get("rows"),
        "size_mb": round(
            details["size_bytes"] / (1024 ** 2), 3
        ),
    }
    for filename, details in artifact_metadata_116.items()
])

final_ground_truth_export_approved_11 = True

print("\n11.6 — EXPORTED ARTIFACT INVENTORY")
display(ground_truth_export_inventory_11)

print("\n" + "=" * 60)
print("NOTEBOOK 11 — FINAL EXPORT & INTEGRITY")
print("=" * 60)
print("Approved attackers       : 400")
print("User-Day GT              : 9,080 rows × 42 columns")
print("Channel-Day GT           : 10,124 rows × 27 columns")
print("Executed attack events   : 29,007")
print("Approved scenario QA     : 35/35 PASS")
print("Channel-Day QA           : 20/20 PASS")
print("User-Day QA              : 27/27 PASS")
print("Independent QA           : 58/58 PASS")
print("Parquet read-back        : PASS")
print("Published SHA-256        : VERIFIED")
print("Original source SHA-256  : UNCHANGED")
print("Existing files overwritten: NO")
print("Data Dictionary          : GENERATED")
print("Ground Truth datasets    : 2")
print("Total exported files     :", len(ALL_NAMES_116))
print("Classification y added   : NO")
print("Features recalculated    : NO")
print("Export location          :", REPORT_11)
print("Final export             : APPROVED")
print("=" * 60)
print("NOTEBOOK 11 — COMPLETE")
print("NEXT: NOTEBOOK 12 — FEATURE ENGINEERING & VALIDATION")



11.6 — EXPORTED ARTIFACT INVENTORY


,filename,rows,size_mb
0,ground_truth_user_days_r62.parquet,9080.0,0.100
1,ground_truth_channel_days_r62.parquet,10124.0,0.087
2,ground_truth_data_dictionary.csv,69.0,0.012
3,scenario_execution_qa.csv,35.0,0.002
4,channel_ground_truth_qa.csv,20.0,0.001
5,user_day_ground_truth_qa.csv,27.0,0.001
6,independent_ground_truth_qa.csv,58.0,0.003
7,notebook_summary.md,NaN,0.002



NOTEBOOK 11 — FINAL EXPORT & INTEGRITY
Approved attackers       : 400
User-Day GT              : 9,080 rows × 42 columns
Channel-Day GT           : 10,124 rows × 27 columns
Executed attack events   : 29,007
Approved scenario QA     : 35/35 PASS
Channel-Day QA           : 20/20 PASS
User-Day QA              : 27/27 PASS
Independent QA           : 58/58 PASS
Parquet read-back        : PASS
Published SHA-256        : VERIFIED
Original source SHA-256  : UNCHANGED
Existing files overwritten: NO
Data Dictionary          : GENERATED
Ground Truth datasets    : 2
Total exported files     : 9
Classification y added   : NO
Features recalculated    : NO
Export location          : C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\11_Ground_Truth_and_Quality_Validation
Final export             : APPROVED
NOTEBOOK 11 — COMPLETE
NEXT: NOTEBOOK 12 — FEATURE ENGINEERING & VALIDATION
